# Validazione end-to-end — Super-risoluzione image-to-image

Walkthrough eseguibile della pipeline `srmri`: degradazione HR→LR, modello
ESPCN, training breve su dati **sintetici** (per restare offline e veloce) e
confronto SR vs bicubica con PSNR/SSIM. Il dataset reale (MedMNIST) è usato dagli
script `scripts/train.py` e `scripts/evaluate.py`.

Riferimenti: ESPCN (Shi et al., CVPR 2016, arXiv:1609.05158), SRCNN (Dong et al.,
TPAMI 2016, DOI 10.1109/TPAMI.2015.2439281), MedMNIST v2 (Yang et al.,
Scientific Data 2023, DOI 10.1038/s41597-022-01721-8).

In [ ]:
import sys
import pathlib

ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import torch

from srmri.data import fake_hr_dataset, fake_loaders
from srmri.degrade import downsample
from srmri.model import build_model
from srmri.baseline import bicubic_sr
from srmri.metrics import psnr, ssim
from srmri.train import train, get_device

device = get_device("cpu")
print("Device:", device)

## 1. Degradazione HR → LR
Generiamo immagini HR sintetiche e le degradiamo per downsample bicubico ×2.

In [ ]:
hr = fake_hr_dataset(n=8, hr_size=32)
lr = downsample(hr, scale=2)
print("HR:", tuple(hr.shape), "LR:", tuple(lr.shape))
assert lr.shape[-1] == hr.shape[-1] // 2

## 2. Training breve della rete ESPCN
Su dati sintetici, poche epoche, solo per verificare la convergenza.

In [ ]:
train_loader, test_loader = fake_loaders(
    n_train=48, n_test=16, hr_size=32, scale=2, batch_size=8
)
model = build_model("espcn", scale=2, channels=1)
out = train(model, train_loader, epochs=30, lr=5e-3, device=device, verbose=False)
losses = out["history"]["train_loss"]
print(f"Loss iniziale: {losses[0]:.5f}  ->  finale: {losses[-1]:.5f}")
assert losses[-1] < losses[0]

## 3. Confronto SR vs bicubica
Calcoliamo PSNR/SSIM sul test set sintetico.

In [ ]:
model.eval()
srs, bics, hrs = [], [], []
with torch.no_grad():
    for lr_b, hr_b in test_loader:
        srs.append(model(lr_b))
        bics.append(bicubic_sr(lr_b, scale=2))
        hrs.append(hr_b)
sr = torch.cat(srs)
bic = torch.cat(bics)
hr_t = torch.cat(hrs)
print(f"PSNR bicubica: {psnr(bic, hr_t):.2f} dB | SR: {psnr(sr, hr_t):.2f} dB")
print(f"SSIM bicubica: {ssim(bic, hr_t):.4f} | SR: {ssim(sr, hr_t):.4f}")

## 4. Visualizzazione qualitativa

In [ ]:
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 3, figsize=(9, 3))
ax[0].imshow(bic[0, 0], cmap="gray", vmin=0, vmax=1)
ax[0].set_title("Bicubica")
ax[1].imshow(sr[0, 0], cmap="gray", vmin=0, vmax=1)
ax[1].set_title("SR (rete)")
ax[2].imshow(hr_t[0, 0], cmap="gray", vmin=0, vmax=1)
ax[2].set_title("HR")
for a in ax:
    a.axis("off")
plt.tight_layout()
print("Figura generata.")

Per i risultati sul **dataset reale** (BloodMNIST 64×64) vedi `results/metrics.json`,
prodotto da `scripts/evaluate.py` dopo `scripts/train.py`.